In [1]:
# --- Clone your empty GitHub repo into Colab ---
# Replace with your actual repo URL
REPO_URL = "https://github.com/iTamojeet/fred-economic-intelligence-agent.git"

!git clone {REPO_URL}
%cd fred-economic-intelligence-agent

# Create the folder structure matching the project spec (Section 28)
import os
for folder in ['data/processed', 'models', 'src', 'app']:
    os.makedirs(folder, exist_ok=True)

print("Repo cloned and folders created")
!ls -la

Cloning into 'fred-economic-intelligence-agent'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/fred-economic-intelligence-agent
Repo cloned and folders created
total 32
drwxr-xr-x 7 root root 4096 Sep  5 12:04 .
drwxr-xr-x 1 root root 4096 Sep  5 12:04 ..
drwxr-xr-x 2 root root 4096 Sep  5 12:04 app
drwxr-xr-x 3 root root 4096 Sep  5 12:04 data
drwxr-xr-x 8 root root 4096 Sep  5 12:04 .git
-rw-r--r-- 1 root root 1065 Sep  5 12:04 LICENSE
drwxr-xr-x 2 root root 4096 Sep  5 12:04 models
drwxr-xr-x 2 root root 4096 Sep  5 12:04 src


In [3]:
# --- Regenerate indicator_dictionary.xlsx (recreate if missing from Drive) ---
import pandas as pd

dictionary_data = [
    {"Indicator": "Unemployment Rate", "FRED Series": "UNRATE", "Frequency": "Monthly", "Unit": "%", "Meaning": "Share of labor force actively seeking work but unemployed", "Why Important": "Core employment health signal; rises sharply in recessions"},
    {"Indicator": "CPI (All Items)", "FRED Series": "CPIAUCSL", "Frequency": "Monthly", "Unit": "Index", "Meaning": "Consumer price level for a basket of goods/services", "Why Important": "Primary measure of inflation"},
    {"Indicator": "Industrial Production Index", "FRED Series": "INDPRO", "Frequency": "Monthly", "Unit": "Index", "Meaning": "Output of manufacturing, mining, and utilities", "Why Important": "Tracks real economic activity/production"},
    {"Indicator": "Federal Funds Rate", "FRED Series": "FEDFUNDS", "Frequency": "Monthly", "Unit": "%", "Meaning": "Interest rate banks charge each other overnight", "Why Important": "Fed's primary monetary policy lever"},
    {"Indicator": "10-Year Treasury Yield", "FRED Series": "DGS10", "Frequency": "Daily", "Unit": "%", "Meaning": "Yield on 10-year US government bonds", "Why Important": "Long-term growth/inflation expectations"},
    {"Indicator": "2-Year Treasury Yield", "FRED Series": "DGS2", "Frequency": "Daily", "Unit": "%", "Meaning": "Yield on 2-year US government bonds", "Why Important": "Short-term rate expectations"},
    {"Indicator": "Real GDP", "FRED Series": "GDPC1", "Frequency": "Quarterly", "Unit": "Billions $ (chained)", "Meaning": "Inflation-adjusted total economic output", "Why Important": "Broadest measure of economic growth"},
    {"Indicator": "NBER Recession Indicator", "FRED Series": "USREC", "Frequency": "Monthly", "Unit": "Binary (0/1)", "Meaning": "Official NBER recession dating", "Why Important": "Ground-truth target for the classification model"},
    {"Indicator": "Yield Curve Spread", "FRED Series": "Derived (DGS10 - DGS2)", "Frequency": "Monthly (avg)", "Unit": "% points", "Meaning": "Long-term minus short-term yield", "Why Important": "Inversion (negative spread) historically precedes recessions"},
    {"Indicator": "Inflation Rate (YoY)", "FRED Series": "Derived (CPIAUCSL pct_change 12)", "Frequency": "Monthly", "Unit": "%", "Meaning": "Year-over-year % change in CPI", "Why Important": "Standard inflation rate used in policy/analysis"},
]

dict_df = pd.DataFrame(dictionary_data)

# Save to BOTH Drive (so it persists for future notebooks) and directly into the repo
dict_df.to_excel('/content/drive/MyDrive/fred-economic-agent/data/indicator_dictionary.xlsx', index=False)
dict_df.to_excel('/content/fred-economic-intelligence-agent/data/indicator_dictionary.xlsx', index=False)

print("Regenerated and saved to both Drive and repo")

Regenerated and saved to both Drive and repo


In [4]:
# --- Copy remaining artifacts: model, features, src files ---
import shutil

DRIVE_BASE = '/content/drive/MyDrive/fred-economic-agent'
REPO_BASE = '/content/fred-economic-intelligence-agent'

shutil.copy(f'{DRIVE_BASE}/models/recession_model.pkl', f'{REPO_BASE}/models/recession_model.pkl')
shutil.copy(f'{DRIVE_BASE}/models/recession_model_features.pkl', f'{REPO_BASE}/models/recession_model_features.pkl')

shutil.copy(f'{DRIVE_BASE}/src/snapshot.py', f'{REPO_BASE}/src/snapshot.py')
shutil.copy(f'{DRIVE_BASE}/src/genai_explain.py', f'{REPO_BASE}/src/genai_explain.py')
shutil.copy(f'{DRIVE_BASE}/src/genai_report.py', f'{REPO_BASE}/src/genai_report.py')

print("Remaining artifacts copied.")
!find {REPO_BASE} -type f -not -path '*/.git/*'

Remaining artifacts copied.
/content/fred-economic-intelligence-agent/models/recession_model.pkl
/content/fred-economic-intelligence-agent/models/recession_model_features.pkl
/content/fred-economic-intelligence-agent/data/indicator_dictionary.xlsx
/content/fred-economic-intelligence-agent/data/processed/economic_dataset.csv
/content/fred-economic-intelligence-agent/LICENSE
/content/fred-economic-intelligence-agent/src/genai_report.py
/content/fred-economic-intelligence-agent/src/genai_explain.py
/content/fred-economic-intelligence-agent/src/snapshot.py


In [8]:
%%writefile /content/fred-economic-intelligence-agent/app/streamlit_app.py
import streamlit as st
import pandas as pd
import joblib
import sys
import os
from google import genai

# --- Path setup so src/ imports work when deployed ---
APP_DIR = os.path.dirname(os.path.abspath(__file__))
ROOT_DIR = os.path.dirname(APP_DIR)
sys.path.append(os.path.join(ROOT_DIR, 'src'))

from snapshot import generate_economic_snapshot
from genai_explain import explain_economic_snapshot
from genai_report import generate_executive_report

st.set_page_config(page_title="Economic Intelligence Agent", layout="wide")

# --- Load data and model (cached so it doesn't reload on every interaction) ---
@st.cache_data
def load_data():
    df = pd.read_csv(os.path.join(ROOT_DIR, 'data/processed/economic_dataset.csv'),
                      index_col='Date', parse_dates=True)
    return df

@st.cache_resource
def load_model():
    model = joblib.load(os.path.join(ROOT_DIR, 'models/recession_model.pkl'))
    features = joblib.load(os.path.join(ROOT_DIR, 'models/recession_model_features.pkl'))
    return model, features

df = load_data()
model, features = load_model()
snapshot = generate_economic_snapshot(df, model, features)

# --- Gemini client setup (API key from Streamlit secrets, not hardcoded) ---
GEMINI_API_KEY = st.secrets.get("GEMINI_API_KEY", None)
client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

# --- Page: Dashboard ---
st.title("Economic Intelligence Agent")
st.caption(f"Latest data as of {snapshot['date']}")

col1, col2, col3, col4 = st.columns(4)
col1.metric("Recession Risk", snapshot['recession_risk'],
            f"{snapshot['recession_probability']*100:.1f}% probability")
col2.metric("Inflation", f"{snapshot['inflation_rate']}%", snapshot['inflation_trend'])
col3.metric("Unemployment", f"{snapshot['unemployment_rate']}%", snapshot['unemployment_trend'])
col4.metric("Yield Curve", snapshot['yield_curve_status'], f"{snapshot['yield_curve_spread']} pp")

st.divider()

# --- Trends chart ---
st.subheader("Economic Trends")
chart_option = st.selectbox("Select indicator", [
    'Unemployment Rate', 'Inflation Rate', 'Industrial Production Growth', 'Yield Curve Spread'
])
st.line_chart(df[chart_option])

st.divider()

# --- GenAI Explanation ---
st.subheader("Ask the Economic Agent")
if client is None:
    st.warning("GEMINI_API_KEY not configured in Streamlit secrets. Explanation features disabled.")
else:
    if st.button("Explain current recession risk"):
        with st.spinner("Generating explanation..."):
            explanation = explain_economic_snapshot(snapshot, client)
        st.write(explanation)

    if st.button("Generate Monthly Economic Intelligence Report"):
        with st.spinner("Generating report..."):
            report = generate_executive_report(snapshot, client)
        st.markdown(report)

Overwriting /content/fred-economic-intelligence-agent/app/streamlit_app.py


In [9]:
%%writefile /content/fred-economic-intelligence-agent/requirements.txt
streamlit
pandas
scikit-learn
joblib
google-genai
openpyxl

Writing /content/fred-economic-intelligence-agent/requirements.txt


In [10]:
# --- Configure git identity (first time only) ---
!git config --global user.email "your_email@example.com"
!git config --global user.name "Your Name"

In [11]:
%cd /content/fred-economic-intelligence-agent

!git add .
!git commit -m "Add Level 1 pipeline: data, model, GenAI, Streamlit app"
!git push

/content/fred-economic-intelligence-agent
[main b43be55] Add Level 1 pipeline: data, model, GenAI, Streamlit app
 9 files changed, 786 insertions(+)
 create mode 100644 app/streamlit_app.py
 create mode 100644 data/indicator_dictionary.xlsx
 create mode 100644 data/processed/economic_dataset.csv
 create mode 100644 models/recession_model.pkl
 create mode 100644 models/recession_model_features.pkl
 create mode 100644 requirements.txt
 create mode 100644 src/genai_explain.py
 create mode 100644 src/genai_report.py
 create mode 100644 src/snapshot.py
fatal: could not read Username for 'https://github.com': No such device or address


In [12]:
%cd /content/fred-economic-intelligence-agent

# Replace YOUR_TOKEN, YOUR_USERNAME, and YOUR_REPO below
!git remote set-url origin https://YOUR_TOKEN@github.com/YOUR_USERNAME/fred-economic-intelligence-agent.git

!git push

/content/fred-economic-intelligence-agent
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (14/14), done.
Writing objects: 100% (16/16), 40.91 KiB | 1.17 MiB/s, done.
Total 16 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), done.
To https://github.com/iTamojeet/fred-economic-intelligence-agent.git
   d79feaa..b43be55  main -> main


In [13]:
import os
os.makedirs('/content/fred-economic-intelligence-agent/notebooks', exist_ok=True)

In [15]:
# --- Locate your Colab notebooks in Drive ---
!find "/content/drive/MyDrive/Colab Notebooks" -iname "*.ipynb" 2>/dev/null

/content/drive/MyDrive/Colab Notebooks/pandasaidemo.ipynb
/content/drive/MyDrive/Colab Notebooks/TMC_SaraswatiPujo.ipynb
/content/drive/MyDrive/Colab Notebooks/coding.ipynb
/content/drive/MyDrive/Colab Notebooks/interview.ipynb
/content/drive/MyDrive/Colab Notebooks/01_data_collection.ipynb
/content/drive/MyDrive/Colab Notebooks/02_eda.ipynb
/content/drive/MyDrive/Colab Notebooks/03_recession_model.ipynb
/content/drive/MyDrive/Colab Notebooks/04_genai_explanation.ipynb
/content/drive/MyDrive/Colab Notebooks/05_streamlit_app.ipynb
